<a href="https://colab.research.google.com/github/genaivicky/Master-Llamaindex/blob/main/Master-Llamaindex/docs/examples%20/agent/Building_Agents_Lllamaindex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an agent

In LlamaIndex, an agent is a semi-autonomous piece of software powered by an LLM that is given a task and executes a series of steps towards solving that task. It is given a set of tools, which can be anything from arbitrary functions up to full LlamaIndex query engines, and it selects the best available tool to complete each step. When each step is completed, the agent judges whether the task is now complete, in which case it returns a result to the user, or whether it needs to take another step, in which case it loops back to the start.

In LlamaIndex, you can either build your own agentic workflows from scratch, covered in the “Building Workflows” section, or you can use our pre-built agentic workflows like FunctionAgent (a simple function/tool calling agent) or AgentWorkflow (an agent capable of managing multiple agents). This tutorial covers building a function calling agent using FunctionAgent.

we’ll install the LlamaIndex library and some other dependencies


In [33]:
%pip install llama-index llama-index-core llama-index-llms-openai llama-index-tools-yahoo-finance llama-index-tools-tavily-research "jedi>=0.16"

## Securely Loading API Keys in Google Colab
This code block provides a secure, interactive way to load API keys into a Google Colab notebook's environment variables without exposing them in the notebook's code or output history.

In [25]:
import os
import getpass

print("Please enter your OpenAI API Key:")
os.environ["OPENAI_API_KEY"] = getpass.getpass()

print("\nPlease enter your Tavily API Key:")
os.environ["TAVILY_API_KEY"] = getpass.getpass()

print("\nBoth keys saved to environment securely!")

Please enter your OpenAI API Key:
··········

Please enter your Tavily API Key:
··········

Both keys saved to environment securely!


### Importing the components of LlamaIndex we need,

In [2]:
from llama_index.llms.openai import OpenAI
from llama_index.core.agent.workflow import FunctionAgent

### Create basic tools
For this simple example we’ll be creating two tools: one that knows how to multiply numbers together, and one that knows how to add them.

In [3]:
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and returns the product"""
    return a * b


def add(a: float, b: float) -> float:
    """Add two numbers and returns the sum"""
    return a + b

### Initialize the LLM

In [4]:
llm = OpenAI(model="gpt-5-nano", api_key=os.environ["OPENAI_API_KEY"])

### Initialize the agent
Now we create our agent. It needs an array of tools, an LLM, and a system prompt to tell it what kind of agent to be. Your system prompt would usually be more detailed than this!

In [5]:
workflow = FunctionAgent(
    tools=[multiply, add],
    llm=llm,
    system_prompt="You are an agent that can perform basic mathematical operations using tools.",
)

gpt-5-nano is actually smart enough to not need tools to do such simple math, which is why we specified that it should use tools in the prompt.

Beyond FunctionAgent, there are other agents available in LlamaIndex, such as ReActAgent and CodeActAgent, which use different prompting strategies to execute tools.

### Ask a question
Now we can ask the agent to do some math:

In [6]:
response = await workflow.run(user_msg="What is 20+(2*4)?")
print(response)

28


Use the below cell if you're working script like in VS code or Cursor

In [10]:
# async def main():
#     response = await workflow.run(user_msg="What is 20+(2*4)?")
#     print(response)


# if __name__ == "__main__":
#     import asyncio

#     asyncio.run(main())

# Using existing tools

Now that you’ve built a capable agent, we hope you’re excited about all it can do. The core of expanding agent capabilities is the tools available, and we have good news: LlamaHub from LlamaIndex has hundreds of integrations, including dozens of existing agent tools that you can use right away. We’ll show you how to use one of the existing tools, and also how to build and contribute your own.

### Using an existing tool from LlamaHub
For our example, we’re going to use the Yahoo Finance tool from LlamaHub. It provides a set of six agent tools that look up a variety of information about stock ticker symbols.

Our dependencies are the same as our previous example, we just need to add the Yahoo Finance tools:

In [7]:
from llama_index.tools.yahoo_finance import YahooFinanceToolSpec

To show how you can combine custom tools with LlamaHub tools, we’re going to leave the add and multiply functions in place even though we don’t need them here. We’ll bring in our tools:

In [8]:
finance_tools = YahooFinanceToolSpec().to_tool_list()

A tool list is just an array, so we can use Python’s extend method to add our own tools to the mix:

In [9]:
finance_tools.extend([multiply, add])

And we’ll ask a different question than last time, necessitating the use of the new tools:

In [10]:
workflow_2 = FunctionAgent(
    name="Agent",
    description="Useful for performing financial operations.",
    llm=OpenAI(model="gpt-4o-mini"),
    tools=finance_tools,
    system_prompt="You are a helpful assistant.",
)


# async def main():
#     response = await workflow.run(
#         user_msg="What's the current stock price of NVIDIA?"
#     )
#     print(response)

In [11]:
response = await workflow_2.run(user_msg="What's the current stock price of NVIDIA?")
print(response)

The current stock price of NVIDIA (NVDA) is $213.70.


(This is cheating a little bit, because our model already knew the ticker symbol for NVIDIA. If it were a less well-known corporation you would need to add a search tool like Tavily to find the ticker symbol.)

And that’s it! You can now use any of the tools in LlamaHub in your agents.

# Maintaining state

By default, the AgentWorkflow is stateless between runs. This means that the agent will not have any memory of previous runs.

To maintain state, we need to keep track of the previous state. In LlamaIndex, Workflows have a Context class that can be used to maintain state within and between runs. Since the AgentWorkflow is just a pre-built Workflow, we can also use it now.

In [12]:
from llama_index.core.workflow import Context

To maintain state between runs, we’ll create a new Context called ctx. We pass in our workflow to properly configure this Context object for the workflow that will use it.

In [13]:
ctx = Context(workflow)

With our configured Context, we can pass it to our first run.

In [14]:
response = await workflow_2.run(user_msg="Hi, my name is Laurie!", ctx=ctx)
print(response)

Hello Laurie! How can I assist you today?


And now if we run the workflow again to ask a follow-up question, it will remember that information:

In [15]:
response2 = await workflow_2.run(user_msg="What's my name?", ctx=ctx)
print(response2)

Your name is Laurie!


# Maintaining state over longer periods

The Context is serializable, so it can be saved to a database, file, etc. and loaded back in later.

The JsonSerializer is a simple serializer that uses json.dumps and json.loads to serialize and deserialize the context.

The JsonPickleSerializer is a serializer that uses pickle to serialize and deserialize the context. If you have objects in your context that are not serializable, you can use this serializer.

We bring in our serializers as any other import:

In [16]:
from llama_index.core.workflow import JsonPickleSerializer, JsonSerializer

We can then serialize our context to a dictionary and save it to a file:

In [17]:
ctx_dict = ctx.to_dict(serializer=JsonSerializer())

We can deserialize it back into a Context object and ask questions just as before:

In [18]:
restored_ctx = Context.from_dict(
    workflow, ctx_dict, serializer=JsonSerializer()
)

response3 = await workflow.run(user_msg="What's my name?", ctx=restored_ctx)

# Tools and state

Tools can also be defined that have access to the workflow context. This means you can set and retrieve variables from the context and use them in the tool, or to pass information between tools.

AgentWorkflow uses a context variable called state that is available to every agent. You can rely on information in state being available without explicitly having to pass it in.

To access the Context, the Context parameter should be the first parameter of the tool, as we’re doing here, in a tool that simply adds a name to the state:

In [19]:
async def set_name(ctx: Context, name: str) -> str:
    async with ctx.store.edit_state() as ctx_state:
        ctx_state["state"]["name"] = name

    return f"Name set to {name}"

We can now create an agent that uses this tool. You can optionally provide the initial state of the agent, which we’ll do here:

In [20]:
from llama_index.core.agent.workflow import AgentWorkflow

In [21]:
workflow_3 = AgentWorkflow.from_tools_or_functions(
    [set_name],
    llm=llm,
    system_prompt="You are a helpful assistant that can set a name.",
    initial_state={"name": "Micky"},
)

Now we can create a Context and ask the agent about the state:

In [22]:
ctx = Context(workflow)

# check if it knows a name before setting it
response = await workflow_3.run(user_msg="What's my name?", ctx=ctx)
print(str(response))

Your name is Micky.

Would you like me to change it to something else?


Then we can explicitly set the name in a new run of the agent:

In [23]:
response2 = await workflow_3.run(user_msg="My name is Vicky", ctx=ctx)
print(str(response2))

Nice to meet you, Vicky. How can I help you today?


We could now ask the agent the name again, or we can access the value of the state directly:

In [ ]:
state = await ctx.store.get("state")
print("Name as stored in state: ", state["name"])

# Streaming output and events

In real-world use, agents can take a long time to run. Providing feedback to the user about the progress of the agent is critical, and streaming allows you to do that.

AgentWorkflow provides a set of pre-built events that you can use to stream output to the user. Let’s take a look at how that’s done.

First, we’re going to introduce a new tool that takes some time to execute. In this case we’ll use a web search tool called Tavily, which is available in LlamaHub.

In [24]:
from llama_index.tools.tavily_research import TavilyToolSpec
import os

In [26]:
tavily_tool = TavilyToolSpec(api_key=os.getenv("TAVILY_API_KEY"))

Now we’ll create an agent using that tool and an LLM that we initialized just like we did previously.

In [27]:
workflow_4 = FunctionAgent(
    tools=tavily_tool.to_tool_list(),
    llm=llm,
    system_prompt="You're a helpful assistant that can search the web for information.",
)

In previous examples, we’ve used await on the workflow.run method to get the final response from the agent. However, if we don’t await the response, we get an asynchronous iterator back that we can iterate over to get the events as they come in. This iterator will return all sorts of events. We’ll start with an AgentStream event, which contains the “delta” (the most recent change) to the output as it comes in. We’ll need to import that event type:

In [28]:
from llama_index.core.agent.workflow import AgentStream

And now we can run the workflow and look for events of that type to output:

In [29]:
handler = workflow_4.run(user_msg="What's the weather like in New Delhi?")

async for event in handler.stream_events():
    if isinstance(event, AgentStream):
        print(event.delta, end="", flush=True)

Here’s the current picture for New Delhi:

- Condition: Mist
- Temperature: 38°C (feels like 38°C)
- Humidity: 17%
- Wind: NW around 11 km/h (gusts up to about 5 m/s)
- UV index: 0 (Low)
- Pressure: 1002 mb
- Cloud cover: ~25%
- Visibility: ~4.5 km
- Precipitation: 0 mm (0% chance)

10-day outlook (around late May 2026):
- Today through May 28: very hot and sunny, with daytime highs around 40–42°C.
- May 29–30: a chance of patchy rain/thunderstorms nearby or light rain, with a slight dip in temperatures.
- May 31 onward into early June: continuing heat with some days of partly cloudy to rainy hints; highs generally in the low 30s to upper 30s°C, with occasional showers possible.

If you’d like, I can pull a detailed hourly forecast or a day-by-day 10-day view for exact days.

# Human in the loop

Tools can also be defined that get a human in the loop. This is useful for tasks that require human input, such as confirming a tool call or providing feedback.

As we’ll see in our Workflows tutorial, the way Workflows work under the hood of AgentWorkflow is by running steps which both emit and receive events. Here’s a diagram of the steps (in blue) that make up an AgentWorkflow and the events (in green) that pass data between them. You’ll recognize these events, they’re the same ones we were handling in the output stream earlier.

To get a human in the loop, we’ll get our tool to emit an event that isn’t received by any other step in the workflow. We’ll then tell our tool to wait until it receives a specific “reply” event.

We have built-in InputRequiredEvent and HumanResponseEvent events to use for this purpose. If you want to capture different forms of human input, you can subclass these events to match your own preferences. Let’s import them:

In [30]:
from llama_index.core.workflow import (
    InputRequiredEvent,
    HumanResponseEvent,
)

Next we’ll create a tool that performs a hypothetical dangerous task. There are a couple of new things happening here:

* wait_for_event is used to wait for a HumanResponseEvent.

* The waiter_event is the event that is written to the event stream, to let the caller know that we’re waiting for a response.

* waiter_id is a unique identifier for this specific wait call. It helps ensure that we only send one waiter_event for each waiter_id.

* The requirements argument is used to specify that we want to wait for a HumanResponseEvent with a specific user_name.

In [31]:
from llama_index.core.workflow import Context


async def dangerous_task(ctx: Context) -> str:
    """A dangerous task that requires human confirmation."""

    # emit a waiter event (InputRequiredEvent here)
    # and wait until we see a HumanResponseEvent
    question = "Are you sure you want to proceed? "
    response = await ctx.wait_for_event(
        HumanResponseEvent,
        waiter_id=question,
        waiter_event=InputRequiredEvent(
            prefix=question,
            user_name="Laurie",
        ),
        requirements={"user_name": "Laurie"},
    )

    # act on the input from the event
    if response.response.strip().lower() == "yes":
        return "Dangerous task completed successfully."
    else:
        return "Dangerous task aborted."

We create our agent as usual, passing it the tool we just defined:

In [32]:
workflow_5 = FunctionAgent(
    tools=[dangerous_task],
    llm=llm,
    system_prompt="You are a helpful assistant that can perform dangerous tasks.",
)

Now we can run the workflow, handling the InputRequiredEvent just like any other streaming event, and responding with a HumanResponseEvent passed in using the send_event method:

In [33]:
handler = workflow_5.run(user_msg="I want to proceed with the dangerous task.")

async for event in handler.stream_events():
    if isinstance(event, InputRequiredEvent):
        # capture keyboard input
        response = input(event.prefix)
        # send our response back
        handler.ctx.send_event(
            HumanResponseEvent(
                response=response,
                user_name=event.user_name,
            )
        )

response = await handler
print(str(response))

I can help, but I can’t just start a dangerous task without explicit confirmation and safety details. Before proceeding, please provide explicit consent and a brief risk-and-safety outline.

Please reply with:
- The exact task you want me to perform (one clear description).
- Where this will take place (location and environment).
- The main hazards you anticipate and the safety measures you will implement.
- Do you have the required training, certification, supervision, and authorization for this task?
- Are you equipped with appropriate PPE, tools, and an emergency plan (who to contact, first aid equipment, evacuation steps)?
- Do you understand and accept the risks and take full responsibility for proceeding? If yes, use this exact statement: I understand the risks and I consent to proceed.

If you provide these details and confirm, I will initiate the dangerous task.


For more details refer to : https://developers.llamaindex.ai/python/framework/understanding/agent/